# 🧬 SARS-CoV-2 Mutation Tracker — Analysis Notebook

This notebook walks through the full analysis pipeline step by step, with interactive
visualisations for each research question.

**Research questions:**
1. Which spike mutations rose to fixation fastest, and when?
2. Is there geographic clustering — did certain mutations dominate specific regions first?
3. How do Alpha → Delta → Omicron transitions look as mutation frequency shifts?
4. Which positions in the RBD mutated most frequently (mutational hotspots)?
5. Which mutations show convergent evolution across independent lineages?
6. How strongly are mutations co-linked (linkage disequilibrium)?

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.io as pio
pio.renderers.default = 'iframe'   # works in JupyterLab; change to 'notebook' for classic

print('Imports OK')

## Step 1 — Generate / load data

First run: generates 5,000 synthetic sequences that faithfully reproduce the real
variant frequency schedule. Replace with Nextstrain or GISAID data for real results.

In [ ]:
from config import OUTPUT_DIR

mut_table = OUTPUT_DIR / 'mutation_table.parquet'

if not mut_table.exists():
    print('Running pipeline for the first time …')
    from pipeline.fetch_data import _generate_sample_metadata
    from pipeline.alignment  import build_mutation_table
    import tempfile, pathlib, config as cfg
    
    meta = _generate_sample_metadata(n=5000)
    df   = build_mutation_table(meta)
else:
    df = pd.read_parquet(mut_table)
    df['date'] = pd.to_datetime(df['date'])

print(f'Loaded {len(df):,} sequences')
print(f'Date range: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'WHO variants: {sorted(df["who_label"].unique())}')
df.head(3)

## Step 2 — Variant dominance over time (Research Q3)

In [ ]:
from analysis.frequency_analysis import compute_variant_frequency
from visualization.plots import timeline_plot

df['year_month'] = df['date'].dt.to_period('M')
vf = compute_variant_frequency(df)

fig = timeline_plot(vf, mode='variant')
fig.show()

## Step 3 — Per-mutation frequency curves (Research Q1)

In [ ]:
from analysis.frequency_analysis import compute_mutation_frequency_over_time

KEY = ['D614G', 'N501Y', 'E484K', 'L452R', 'T478K', 'P681H', 'K417N']
mf = compute_mutation_frequency_over_time(df, mutations=KEY)

fig = timeline_plot(vf, mutation_freq=mf, mode='mutation', selected_mutations=KEY)
fig.show()

print('\nPeak frequencies:')
print(mf.groupby('mutation')['frequency'].max().sort_values(ascending=False).round(1))

## Step 4 — Sweep speed analysis (Research Q1)

In [ ]:
from analysis.frequency_analysis import compute_sweep_speed
from visualization.plots import sweep_speed_chart

sweeps = compute_sweep_speed(df)
print(sweeps.to_string(index=False))

sweep_speed_chart(sweeps).show()

## Step 5 — Geographic spread (Research Q2)

In [ ]:
from analysis.frequency_analysis import compute_geographic_spread
from visualization.plots import geographic_plotly

geo = compute_geographic_spread(df)

# First detection per variant
print('First detection dates by variant:')
first = geo.sort_values('first_detection').drop_duplicates('who_label')
print(first[['who_label','country','first_detection']].to_string(index=False))

geographic_plotly(geo).show()

## Step 6 — Spike protein hotspot map (Research Q4)

In [ ]:
from analysis.frequency_analysis import compute_mutation_hotspots
from visualization.plots import spike_protein_map, spike_rbd_detail

hs = compute_mutation_hotspots(df)

print(f'Total mutated positions: {len(hs)}')
print(f'Hotspot positions (top 10%): {hs["is_hotspot"].sum()}')
print('\nTop 10 positions:')
print(hs.nlargest(10,'mutation_count')[['position','domain','mutation_count','top_mutations']].to_string(index=False))

spike_protein_map(hs).show()
spike_rbd_detail(hs).show()

## Step 7 — Convergent evolution (Research Q5)

In [ ]:
from analysis.frequency_analysis import detect_convergent_evolution

conv = detect_convergent_evolution(df)

print(f'Convergent mutations found: {len(conv)}')
print('\nTop convergent mutations (appeared independently in most variants):')
print(conv.nlargest(10,'appearance_count').to_string(index=False))

# Visualise
import plotly.express as px
if not conv.empty:
    fig = px.bar(
        conv.nlargest(15,'appearance_count'),
        x='mutation', y='appearance_count',
        color='appearance_count',
        color_continuous_scale='Reds',
        title='Convergent spike mutations — number of independent variant appearances',
        labels={'mutation':'Mutation','appearance_count':'Variants carrying this mutation'},
    )
    fig.update_layout(coloraxis_showscale=False)
    fig.show()

## Step 8 — Co-occurrence / linkage disequilibrium (Research Q6)

In [ ]:
from analysis.frequency_analysis import compute_mutation_cooccurrence
from visualization.plots import cooccurrence_heatmap

cooccur = compute_mutation_cooccurrence(df, top_n=15)

print('Top co-occurring mutation pairs (by odds ratio):')
print(cooccur.head(15)[['mut_a','mut_b','cooccurrence_count','odds_ratio','p_value']].to_string(index=False))

cooccurrence_heatmap(cooccur).show()

## Step 9 — Phylogenetic tree

In [ ]:
from visualization.plots import phylogenetic_tree_plotly
phylogenetic_tree_plotly().show()

## Step 10 — Summary statistics

In [ ]:
import json

print('=' * 55)
print('SARS-CoV-2 MUTATION TRACKER — SUMMARY STATISTICS')
print('=' * 55)
print(f'Sequences analysed:        {len(df):,}')
print(f'Date range:                {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Countries:                 {df["country"].nunique()}')
print(f'WHO variants:              {df["who_label"].nunique()}')

all_muts = set()
for m in df['spike_mutations'].dropna():
    if isinstance(m, str):
        all_muts.update(json.loads(m))
print(f'Unique spike mutations:    {len(all_muts):,}')
print(f'Convergent mutations:      {len(conv)}')
print(f'RBD hotspot positions:     {len(hs[hs["domain"].isin(["RBD","RBM"])])}')

if not sweeps.empty:
    fastest = sweeps.dropna(subset=['sweep_days']).nsmallest(1,'sweep_days')
    if not fastest.empty:
        r = fastest.iloc[0]
        print(f'Fastest sweeping mutation: {r["mutation"]} ({int(r["sweep_days"])} days)')
print('=' * 55)